# 01 — Inventory: load every model (PLAN.md §10, §13.2)
For each active model in `configs/models.yaml`: load it, answer one benign prompt greedily, report VRAM, free memory.
RL is run twice: with its guard chat template (BASE weights + guard) and with BASE's template (guard removed).

Requires the Kaggle secret `HF_TOKEN` and a T4 GPU.

In [ ]:
# (a) Clone the repo (or update it when re-running in the same session). Non-interactive.
import os, subprocess
REPO_URL = "https://github.com/Bennytimz/aftermarket"
if os.path.basename(os.getcwd()) != "aftermarket":
    if not os.path.isdir("aftermarket/.git"):
        subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    os.chdir("aftermarket")
subprocess.run(["git", "pull", "-q", "--ff-only"], check=False)
print("repo at", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
# (b) Install only what Kaggle lacks (requirements-kaggle.txt), keeping Kaggle's torch build fixed.
#     torchao is removed: Kaggle's 0.10.0 breaks PEFT imports and we don't use it.
!pip uninstall -y -q torchao
!pip freeze | grep -iE '^(torch|torchvision|torchaudio)==' > /tmp/keep-torch.txt
!pip install -q -r requirements-kaggle.txt -c /tmp/keep-torch.txt
import importlib.metadata as md
for p in ["torch", "transformers", "peft", "accelerate", "huggingface_hub", "datasets", "bitsandbytes",
          "hqq", "curatorkit", "safetune", "circuitkit", "auditkit"]:
    try: print(f"{p:16s}", md.version(p))
    except md.PackageNotFoundError: print(f"{p:16s} MISSING")

In [ ]:
# (c) Log in to Hugging Face from Kaggle secrets (works in background "Save & Run All" runs; never prompts).
# Attach the secret HF_TOKEN (Add-ons -> Secrets) before committing.
from huggingface_hub import login, whoami
HF_USER = None
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)
    HF_USER = whoami()["name"]
    print("HF user:", HF_USER)
except Exception as e:
    print(f"WARNING: no HF login ({type(e).__name__}: {e}). Gated models will fail; pushes/pulls are skipped.")

In [ ]:
# (d) GPU info
!nvidia-smi
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1e9:.1f} GB")

In [ ]:
# (e) Pinned model revision
BASE_MODEL = "CohereLabs/tiny-aya-global"
BASE_REVISION = "af89d219b53ed9b13b8a4645f9c8028973510324"

In [ ]:
import sys, time, subprocess
sys.path.insert(0, "src")
import pandas as pd, torch
from aftermarket.load import load_model, generate, free_memory, active_model_ids, load_registry

PRECISION = "bf16"   # or "4bit" / "fp16"
PROMPT = [{"role": "user", "content": "How do I open a savings account?"}]
assert load_registry()["base_revision"] == BASE_REVISION

def gpu_used_gb():
    out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits", "-i", "0"],
                         capture_output=True, text=True).stdout.strip()
    return round(int(out) / 1024, 2) if out else None

runs = [(m, "native") for m in active_model_ids()] + [("RL", "base")]
print(runs)

In [ ]:
rows = []
for mid, template in runs:
    label = mid if template == "native" else f"{mid}[{template}-template]"
    print(f"\n===== {label}")
    free_memory()
    t0 = time.time()
    try:
        model, tok = load_model(mid, precision=PRECISION, template=template)
        load_s = time.time() - t0
        t1 = time.time()
        reply = generate(model, tok, PROMPT, max_new_tokens=256)
        gen_s = time.time() - t1
        row = dict(model=label, ok=True, load_s=round(load_s, 1), gen_s=round(gen_s, 1),
                   torch_peak_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2), nvsmi_used_gb=gpu_used_gb(),
                   reply=reply, error="")
        print(reply)
        del model, tok
    except Exception as e:
        row = dict(model=label, ok=False, load_s=round(time.time() - t0, 1), error=f"{type(e).__name__}: {e}"[:500])
        print("FAILED:", row["error"])
    print({k: v for k, v in row.items() if k != "reply"})
    rows.append(row)
    free_memory()

In [ ]:
# Save results (failures here don't stop the summary below)
import os, json, datetime
try:
    df = pd.DataFrame(rows)
    run_dir = f"results/01_inventory_{datetime.datetime.utcnow():%Y%m%d_%H%M%S}"
    os.makedirs(run_dir, exist_ok=True)
    df.to_parquet(f"{run_dir}/inventory.parquet")
    sha = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    json.dump({"git_sha": sha, "precision": PRECISION, "base_revision": BASE_REVISION, "prompt": PROMPT,
               "time_utc": datetime.datetime.utcnow().isoformat()}, open(f"{run_dir}/manifest.json", "w"), indent=1)
    print("saved", run_dir)
except Exception as e:
    print("SAVE FAILED:", type(e).__name__, e)

Any model that fails here and can't be fixed within 30 minutes: set `status: dropped` with a `drop_reason` in `configs/models.yaml` (PLAN.md §3).

In [ ]:
# Summary (always runs)
import pandas as pd
summary = pd.DataFrame(rows if "rows" in globals() else [])
cols = [c for c in ["model", "ok", "load_s", "gen_s", "torch_peak_gb", "nvsmi_used_gb", "error"] if c in summary]
with pd.option_context("display.max_colwidth", 120, "display.width", 200):
    print(summary[cols].to_string(index=False) if len(summary) else "no runs recorded")
if len(summary):
    print("\nOK:", list(summary.loc[summary.ok, "model"]), "| FAILED:", list(summary.loc[~summary.ok, "model"]))